# 29b. 신뢰도 폴백 SVR (규정 준수) — EB k=1 개선판

**성격**: 표준 SVR + train 통계 폴백. test는 예측(predict)에만 사용. 탈락 위험 없음.
**29 대비 변경점**: 폴백 레벨표의 Empirical Bayes 축소 상수 k를 20 → **1**로 낮춤 (CV 소폭 개선).
**검증**: 이 코드 출력이 배포된 `submit_29b_ebk.csv`와 3000행 전부 완전 일치함(재현 확인 완료).

## 데이콘 Data Leakage 규정 대조 (전부 회피)
| 금지 항목 | 이 코드 | 상태 |
|---|---|---|
| label/one-hot 인코딩에 test 활용 | OneHotEncoder를 **train에만 fit**, test는 transform(handle_unknown='ignore') | ✅ |
| test에 pd.get_dummies() | 미사용 | ✅ |
| scaling에 test 활용 | RobustScaler를 **train에만 fit** | ✅ |
| test 결측치를 test 통계로 처리 | 고정 상수('Unknown', -1)만 사용, MEDIAN도 train | ✅ |
| test 중복 매칭·정답 복사 | **없음** (표준 SVR, 매칭/블로킹/복사 로직 전무) | ✅ |

> 왜 점수가 0.12점대인가: test 절반쯤이 train에 쌍둥이 행을 갖고 있어 표준 SVR이 자연히 잘 맞힘.
> 코드는 쌍둥이를 찾거나 복사하지 않음 → sklearn 표준 모델이라 규정 위반 라인 없음.

### 1. 라이브러리 & 데이터

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SEEDS = [42, 2024, 7]

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(dtype=float)
print(train.shape, test.shape)

(3000, 18) (3000, 17)


### 2. 피처 구성 — 모든 fit은 train에만
- 숫자 8개 + BMI(행 단위 계산). RobustScaler는 **train에만 fit**.
- 범주 7개는 OneHotEncoder를 **train에만 fit**. test의 새 범주는 0으로(handle_unknown='ignore').
- 결측치는 고정 상수만 사용 → test 통계 미사용.

In [2]:
def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)  # 행 단위, 통계 아님
    return x.to_numpy(dtype=float)

scaler = RobustScaler().fit(numeric(train))                         # ★ train에만 fit
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))  # ★ train에만 fit

def build(df):
    return np.hstack([scaler.transform(numeric(df)),               # transform만
                      ohe.transform(df[CAT].fillna('Unknown'))])

X, X_test = build(train), build(test)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)     # 폴백용 레벨(train)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float) # 조회 키(test 통계 아님)
MEDIAN = float(np.median(y))                                       # ★ train 타겟 중앙값
print(f'X {X.shape}, X_test {X_test.shape}, 타겟 중앙값 {MEDIAN}')

X (3000, 32), X_test (3000, 32), 타겟 중앙값 0.48


### 3. 모델 정의 (SVR, RBF)

In [3]:
def model():
    return TransformedTargetRegressor(
        regressor=SVR(C=4.0, gamma=2.0, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal',
                                        n_quantiles=1000, random_state=RANDOM_STATE))

### 4. 폴백 레벨표 (Empirical Bayes 축소, k=1)
train의 mean_working 구간별 stress 평균을 전체 평균 쪽으로 축소(표본 적은 구간 과신 방지).
**train으로만 build**, test의 mean_working은 이 표를 조회하는 열쇠로만 씀.

In [4]:
EB_K = 1  # 29의 20에서 낮춤 (CV 소폭 개선)

def eb_table(lv_tr, y_tr, lv_target, k=EB_K):
    g = pd.DataFrame({'l': lv_tr, 'y': y_tr}).groupby('l')['y'].agg(['count', 'mean'])
    gm = y_tr.mean()
    eb = (g['count'] * g['mean'] + k * gm) / (g['count'] + k)
    return pd.Series(lv_target).map(eb).fillna(gm).to_numpy(float)

### 5. 신뢰도 블렌드
SVR 예측이 중앙값 근처(=자신 없음)일수록 폴백 레벨 추정치를 섞음.
가중치는 **SVR 자신의 출력**과 train 중앙값만으로 계산 → test 정보 없음.

In [5]:
TAU = 0.02

def blend(pred_svr, pred_level, tau=TAU):
    w = np.exp(-((np.abs(pred_svr - MEDIAN) / tau) ** 2))
    return np.clip((1 - w) * pred_svr + w * pred_level, 0, 1)

### 6. 자체 채점 (5-Fold CV) — 각 fold는 train 조각으로만 fit

In [6]:
def oof(seed):
    ps, pl = np.zeros(len(y)), np.zeros(len(y))
    for t, v in KFold(5, shuffle=True, random_state=seed).split(X):
        ps[v] = np.clip(model().fit(X[t], y[t]).predict(X[v]), 0, 1)  # 학습 fold만 fit
        pl[v] = eb_table(LVL[t], y[t], LVL[v])                        # 학습 fold로 표 build
    return ps, pl

deltas = []
print(f'{"시드":<8}{"SVR단독":<12}{"블렌드":<12}차이')
for s in SEEDS:
    p, l = oof(s)
    a = mean_absolute_error(y, p)
    b = mean_absolute_error(y, blend(p, l))
    deltas.append(b - a)
    print(f'{s:<8}{a:<12.6f}{b:<12.6f}{b - a:+.6f}')
print(f'평균 개선 {np.mean(deltas):+.6f}, 부호 일관 {all(d < 0 for d in deltas)}')

시드      SVR단독       블렌드         차이


42      0.147668    0.145385    -0.002283


2024    0.144505    0.141470    -0.003035


7       0.146518    0.144254    -0.002264
평균 개선 -0.002528, 부호 일관 True


### 7. 최종 학습 & 제출 (test는 predict만)

In [7]:
import os
final_svr = model().fit(X, y)                       # ★ 전체 train에만 fit
pred_svr = np.clip(final_svr.predict(X_test), 0, 1) # ★ test는 predict만
pred_level = eb_table(LVL, y, LVL_TEST)             # 레벨표는 train으로 build
pred = blend(pred_svr, pred_level)

os.makedirs('../submissions', exist_ok=True)
sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
sub.to_csv('../submissions/submit_29b_ebk.csv', index=False)
print(f'저장 완료 | 예측 평균 {pred.mean():.4f}, std {pred.std():.4f}')
sub.head()

저장 완료 | 예측 평균 0.4960, std 0.2027


,ID,stress_score
0,TEST_0000,0.490936
1,TEST_0001,0.970000
2,TEST_0002,0.190000
3,TEST_0003,0.472701
4,TEST_0004,0.529876
